# 🎯 Notebook 13: Final Synthesis & Investment Thesis

---

**Author:** Dnyanesh  
**Date:** February 2025  
**Series:** Tata Motors Deep Dive (13 of 13)

## Objective

Combine all 12 notebooks' findings into:
1. **Unified signal** combining ML, sentiment, and technical analysis
2. **Investment thesis** with conviction score
3. **Risk assessment** with concrete scenarios
4. **Self-critique** — what we got wrong and limitations

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
PROCESSED_DIR = '../data/processed'
print('✅ Environment ready')

In [ ]:
# Load all outputs
files = {}
for f in ['tata_motors_clean.csv', 'tata_motors_all_features.csv', 'model_comparison.csv', 'prophet_forecast.csv', 'backtest_results.csv', 'strategy_metrics.csv', 'selected_features.csv']:
    fp = os.path.join(PROCESSED_DIR, f)
    if os.path.exists(fp):
        files[f] = pd.read_csv(fp, index_col=0, parse_dates=True) if 'clean' in f or 'features' in f or 'backtest' in f else pd.read_csv(fp, index_col=0)
        print(f'  ✅ {f}: {files[f].shape}')
    else:
        print(f'  ❌ {f}: not found')

---

## Part 1: 12-Notebook Journey Recap

| NB | Topic | Key Finding |
|----|-------|------------|
| 01 | Data Extraction | 5+ years of OHLCV data from yfinance |
| 02 | Data Cleaning | Missing-value imputation, outlier flagging |
| 03 | Technical Features | 20+ technical indicators (RSI, MACD, BB) |
| 04 | Statistical Features | Rolling volatility, z-scores, regime detection |
| 05 | EDA | Trend regimes identified, Oct 2024 crash analyzed |
| 06 | Sentiment | News sentiment has weak but measurable correlation |
| 07 | Clustering | 3-4 market phases identified via K-Means |
| 08 | Model Comparison | RF/XGBoost ~55% accuracy, beats random |
| 09 | Feature Selection | Reduced from 30+ to ~15 features |
| 10 | Hyperparameter Tuning | 1-3% accuracy improvement via Optuna |
| 11 | Prophet Forecasting | Seasonal patterns + changepoint detection |
| 12 | Backtesting | ML strategy: better risk-adjusted returns |

In [ ]:
# Load price data
df = files.get('tata_motors_clean.csv', files.get('tata_motors_all_features.csv'))
if df is not None:
    close = df['Close']
    lookback = min(252, len(close))
    print(f'\nPrice Range: {close.min():.2f} to {close.max():.2f}')
    print(f'Current Price: {close.iloc[-1]:.2f}')
    print(f'{lookback}-day High: {close.iloc[-lookback:].max():.2f}')
    print(f'{lookback}-day Low:  {close.iloc[-lookback:].min():.2f}')
    if lookback > 1:
        print(f'Period Return:   {(close.iloc[-1]/close.iloc[-lookback]-1)*100:+.1f}%')

---

## Part 2: Signal Integration

We combine multiple signals into a composite score:

$$\text{Composite} = w_1 \cdot S_{\text{ML}} + w_2 \cdot S_{\text{tech}} + w_3 \cdot S_{\text{sentiment}} + w_4 \cdot S_{\text{trend}}$$

In [ ]:
signals = {}

# Signal 1: Technical (RSI-based)
if df is not None and 'RSI_14' in df.columns:
    rsi = df['RSI_14'].iloc[-1]
    if rsi < 30: signals['Technical (RSI)'] = {'signal': 'BUY', 'score': 0.8, 'reason': f'RSI={rsi:.0f} (oversold)'}
    elif rsi > 70: signals['Technical (RSI)'] = {'signal': 'SELL', 'score': -0.8, 'reason': f'RSI={rsi:.0f} (overbought)'}
    else: signals['Technical (RSI)'] = {'signal': 'NEUTRAL', 'score': (50-rsi)/50, 'reason': f'RSI={rsi:.0f}'}
else:
    signals['Technical (RSI)'] = {'signal': 'N/A', 'score': 0, 'reason': 'RSI not available'}

# Signal 2: Trend (SMA)
if df is not None:
    sma_50 = close.rolling(50).mean().iloc[-1]
    sma_200 = close.rolling(200).mean().iloc[-1]
    price = close.iloc[-1]
    if price > sma_50 > sma_200:
        signals['Trend (SMA)'] = {'signal': 'BULLISH', 'score': 0.6, 'reason': 'Price > SMA50 > SMA200'}
    elif price < sma_50 < sma_200:
        signals['Trend (SMA)'] = {'signal': 'BEARISH', 'score': -0.6, 'reason': 'Price < SMA50 < SMA200'}
    else:
        signals['Trend (SMA)'] = {'signal': 'MIXED', 'score': 0, 'reason': 'SMAs not aligned'}

# Signal 3: ML Model
if 'model_comparison.csv' in files:
    mc = files['model_comparison.csv']
    best_acc = mc['Accuracy'].max()
    signals['ML Model'] = {'signal': 'MODERATE', 'score': (best_acc - 0.5)*2, 'reason': f'Best accuracy: {best_acc:.3f}'}
else:
    signals['ML Model'] = {'signal': 'N/A', 'score': 0, 'reason': 'Model results not available'}

# Signal 4: Momentum
if df is not None:
    ret_5d = (close.iloc[-1]/close.iloc[-5]-1)*100
    ret_21d = (close.iloc[-1]/close.iloc[-21]-1)*100
    momentum_score = np.clip((ret_21d)/20, -1, 1)
    signals['Momentum'] = {'signal': 'UP' if ret_21d > 0 else 'DOWN', 'score': momentum_score, 'reason': f'5D: {ret_5d:+.1f}%, 21D: {ret_21d:+.1f}%'}

print('SIGNAL DASHBOARD')
print('=' * 60)
for name, info in signals.items():
    print(f'  {name:25s}: {info["signal"]:8s} (score: {info["score"]:+.2f}) — {info["reason"]}')

In [ ]:
# Composite score
scores = [s['score'] for s in signals.values()]
composite = np.mean(scores)
print(f'\nCOMPOSITE SIGNAL: {composite:+.3f}')
if composite > 0.3:
    verdict = '🟢 BULLISH — Favor long positions'
elif composite > 0:
    verdict = '🟡 SLIGHTLY BULLISH — Cautious optimism'
elif composite > -0.3:
    verdict = '🟡 SLIGHTLY BEARISH — Reduce exposure'
else:
    verdict = '🔴 BEARISH — Defensive positioning'
print(f'Verdict: {verdict}')

In [ ]:
# Signal radar chart
fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
labels = list(signals.keys())
values = [s['score'] for s in signals.values()] + [scores[0]]
angles = np.linspace(0, 2*np.pi, len(labels), endpoint=False).tolist() + [0]

ax.plot(angles, values, 'o-', color='#3498DB', linewidth=2)
ax.fill(angles, values, alpha=0.15, color='#3498DB')
ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels, fontsize=10)
ax.set_ylim(-1, 1)
ax.set_title(f'Signal Radar (Composite: {composite:+.3f})', fontsize=14, fontweight='bold', y=1.08)
ax.axhline(0, color='gray', linestyle='-', alpha=0.3)
plt.tight_layout(); plt.show()

---

## Part 3: Investment Thesis

### Thesis Framework:

| Factor | Assessment | Weight |
|--------|-----------|--------|
| **Growth Drivers** | EV transition, JLR turnaround | High |
| **Risks** | Debt levels, EV competition | Medium |
| **Valuation** | Relative to sector | Medium |
| **Technicals** | Current signals | Low |
| **Sentiment** | Market mood | Low |

In [ ]:
# Conviction scoring
print('INVESTMENT THESIS')
print('=' * 60)

factors = [
    ('EV Transition Potential', 7, 'Tata is India\'s EV leader (70%+ market share in EVs)'),
    ('JLR Turnaround', 6, 'Luxury segment recovering, margins improving'),
    ('Domestic Market Position', 8, 'Strong brand, growing SUV segment'),
    ('Debt Concerns', -4, 'High debt-to-equity, but improving'),
    ('Global Macro Risk', -3, 'EV subsidy uncertainty, commodity prices'),
    ('Competition Intensity', -5, 'Hyundai, MG, BYD entering India EV'),
    ('Technical Setup', int(composite*5), f'Composite signal: {composite:+.3f}'),
]

total_positive = sum(s for _, s, _ in factors if s > 0)
total_negative = sum(s for _, s, _ in factors if s < 0)
net = total_positive + total_negative
conviction = min(max(net / 20 * 100, 0), 100)

for name, score, reason in factors:
    bar = ('█' * abs(score)) if score >= 0 else ('░' * abs(score))
    color = '+' if score >= 0 else '-'
    print(f'  {color} {name:30s}: {score:+3d}  {bar:12s}  {reason}')

print(f'\nNet Score: {net:+d}')
print(f'Conviction: {conviction:.0f}%')

In [ ]:
# Conviction gauge
fig, ax = plt.subplots(figsize=(10, 3))
ax.barh(0, conviction, color='#2ECC71' if conviction > 50 else '#F39C12' if conviction > 30 else '#E74C3C',
       height=0.5, alpha=0.8)
ax.barh(0, 100, color='#ECF0F1', height=0.5, alpha=0.3)
ax.set_xlim(0, 100)
ax.set_yticks([])
ax.set_title(f'Conviction Meter: {conviction:.0f}%', fontweight='bold', fontsize=14)
ax.axvline(50, color='gray', linestyle='--', alpha=0.5)
ax.text(conviction, 0, f' {conviction:.0f}%', va='center', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

---

## Part 4: Risk Scenarios

What could go wrong?

In [ ]:
print('RISK SCENARIO ANALYSIS')
print('=' * 60)

if df is not None:
    price = close.iloc[-1]
    vol_ann = close.pct_change().std() * np.sqrt(252)
    
    scenarios = [
        ('🟢 Bull Case', 1.25, 'EV adoption accelerates, JLR margins expand'),
        ('🟡 Base Case', 1.05, 'Steady growth, sector rotation'),
        ('🟠 Bear Case', 0.80, 'EV competition intensifies, demand slows'),
        ('🔴 Tail Risk', 0.55, 'Global recession + credit crisis'),
    ]
    
    print(f'Current Price: ₹{price:.2f}')
    print(f'Annualized Volatility: {vol_ann*100:.1f}%\n')
    
    for label, mult, reason in scenarios:
        target = price * mult
        ret = (mult - 1) * 100
        print(f'{label}: ₹{target:,.0f} ({ret:+.0f}%)')
        print(f'  Rationale: {reason}\n')

In [ ]:
# Risk visualization
if df is not None:
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.plot(close.index[-lookback:], close.iloc[-lookback:], color='#2C3E50', linewidth=2, label='Actual Price')
    
    last_date = close.index[-1]
    for label, mult, color_s in [('Bull', 1.25, '#2ECC71'), ('Base', 1.05, '#3498DB'),
                                ('Bear', 0.80, '#F39C12'), ('Tail', 0.55, '#E74C3C')]:
        ax.axhline(price*mult, color=color_s, linestyle='--', alpha=0.6, label=f'{label}: ₹{price*mult:,.0f}')
    
    ax.set_title('Price Scenarios (12-Month)', fontweight='bold', fontsize=14)
    ax.legend(loc='upper left', fontsize=10)
    ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

---

## Part 5: Self-Critique & Limitations

Every analysis has blind spots. Here's an honest assessment:

In [ ]:
print('LIMITATIONS & SELF-CRITIQUE')
print('=' * 60)

limitations = [
    ('Survivorship Bias', 'We only analyzed Tata Motors because it survived. Failed companies are invisible.'),
    ('Data Snooping', 'Testing many features on the same data increases false discoveries.'),
    ('Look-Ahead Bias', 'Despite TimeSeriesSplit, feature engineering uses full history.'),
    ('No Transaction Costs', 'Real costs Brokerage, impact cost, slippage are higher.'),
    ('Single Stock', 'No portfolio diversification analysis.'),
    ('Regime Change', 'Models trained on past regimes may fail in new ones.'),
    ('Sentiment Data Quality', 'News scraping is noisy and incomplete.'),
    ('No Fundamental Analysis', 'P/E, debt ratios, management quality not included.'),
]

for i, (title, detail) in enumerate(limitations):
    print(f'  {i+1}. [{title}]')
    print(f'     {detail}\n')

In [ ]:
# What would we do differently?
print('WHAT WE WOULD DO DIFFERENTLY')
print('=' * 60)
improvements = [
    'Use alternative data (satellite, social media, insider trading)',
    'Try deep learning (LSTM, Transformer) for sequence modeling',
    'Add fundamental data (quarterly earnings, balance sheet)',
    'Build a portfolio instead of single-stock analysis',
    'Use intraday data for more granular signals',
    'Implement proper walk-forward with purged cross-validation',
    'Add options-derived features (implied volatility, put-call ratio)',
]
for i, imp in enumerate(improvements):
    print(f'  {i+1}. {imp}')

---

## Part 6: Final Dashboard

In [ ]:
# Final summary dashboard
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Price chart with SMAs
ax = axes[0,0]
if df is not None:
    ax.plot(close.index[-lookback:], close.iloc[-lookback:], color='#2C3E50', linewidth=1.5, label='Close')
    sma50 = close.rolling(50).mean()
    sma200 = close.rolling(200).mean()
    ax.plot(sma50.index[-lookback:], sma50.iloc[-lookback:], color='#3498DB', linewidth=1, label='SMA 50')
    ax.plot(sma200.index[-lookback:], sma200.iloc[-lookback:], color='#E74C3C', linewidth=1, label='SMA 200')
    ax.set_title('Price & Moving Averages', fontweight='bold')
    ax.legend(fontsize=9)

# 2. Return distribution
ax = axes[0,1]
if df is not None:
    rets = close.pct_change().dropna()
    ax.hist(rets*100, bins=50, color='#3498DB', alpha=0.7, edgecolor='white')
    ax.axvline(0, color='black', linewidth=1)
    ax.axvline(rets.mean()*100, color='red', linestyle='--', label=f'Mean: {rets.mean()*100:.3f}%')
    ax.set_title('Daily Return Distribution', fontweight='bold')
    ax.set_xlabel('Return (%)'); ax.legend()

# 3. Signal composite
ax = axes[1,0]
sig_names = list(signals.keys())
sig_scores = [s['score'] for s in signals.values()]
colors_s = ['#2ECC71' if s > 0 else '#E74C3C' for s in sig_scores]
ax.barh(sig_names, sig_scores, color=colors_s, alpha=0.8)
ax.axvline(0, color='black', linewidth=0.5)
ax.set_title(f'Signal Dashboard (Composite: {composite:+.3f})', fontweight='bold')

# 4. Model accuracy
ax = axes[1,1]
if 'model_comparison.csv' in files:
    mc = files['model_comparison.csv']
    mc['Accuracy'].plot(kind='barh', ax=ax, color='#F39C12', alpha=0.8)
    ax.axvline(0.5, color='red', linestyle='--', alpha=0.5, label='Random')
    ax.set_title('Model Accuracy Comparison', fontweight='bold')
    ax.legend()
else:
    ax.text(0.5, 0.5, 'Model results\nnot available', ha='center', va='center', fontsize=14)
    ax.set_title('Model Accuracy', fontweight='bold')

plt.suptitle('TATA MOTORS — FINAL ANALYSIS DASHBOARD', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()

---

## Conclusion

### Verdict:

Tata Motors represents a **compelling but volatile** investment opportunity:

**FOR:**
- India's EV market leader with first-mover advantage
- JLR turnaround providing luxury segment earnings
- Strong domestic brand in growing SUV segment

**AGAINST:**
- High debt levels constrain flexibility
- Intense EV competition from Chinese manufacturers (BYD)
- Global macro uncertainty affects luxury demand

**Our ML Analysis Adds:**
- ~55% directional accuracy — marginally better than random
- Prophet shows seasonal price patterns that can inform entry/exit timing
- Backtesting suggests ML-guided risk management reduces drawdowns
- Feature selection reveals volume and volatility as key predictive signals

### Final Recommendation:

> **Moderate HOLD with tactical trading overlay.** Use ML signals to adjust position size (not for binary buy/sell decisions). Combine with fundamental analysis for a complete picture.

### Confidence Level: MODERATE

The stock market is inherently unpredictable. All models are wrong; some are useful.

---

*🎓 End of Tata Motors Deep Dive Series*